<a href="https://colab.research.google.com/github/robertocarter1568/Roberto/blob/master/RandomSpotify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install spotipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.2/354.2 kB 11.3 MB/s eta 0:00:00


In [7]:
import random
import spotipy
from spotipy.oauth2 import SpotifyOAuth

# ========= CONFIG =========
CLIENT_ID = "595410af6921445c83ddeb91a7ae2089"
CLIENT_SECRET = "13d8e4b30f594e98aa21e7d1a7d9f15f"
REDIRECT_URI = "https://example.com/callback"  # misma que pusiste en Spotify
SCOPE = "playlist-modify-private"
ARTIST_FILE = "artistas.txt"
PLAYLIST_NAME = "Random desde TXT 🎲"


# ========= AUTH =========
sp = spotipy.Spotify(
    auth_manager=SpotifyOAuth(
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
        redirect_uri=REDIRECT_URI,
        scope=SCOPE,
        open_browser=False
    )
)

# ========= INPUT =========
num_artistas = int(input("¿Cuántos artistas random tomar? "))
canciones_por_artista = int(input("¿Cuántas canciones random por artista? "))

# ========= LEER ARTISTAS =========
with open(ARTIST_FILE, "r", encoding="utf-8") as f:
    artistas = [a.strip() for a in f if a.strip()]

artistas_random = random.sample(artistas, min(num_artistas, len(artistas)))
tracks = []

# ========= RANDOM MUSICAL =========
for nombre in artistas_random:
    result = sp.search(q=f"artist:{nombre}", type="artist", limit=1)
    if not result["artists"]["items"]:
        continue

    artist_id = result["artists"]["items"][0]["id"]

    albums = sp.artist_albums(
        artist_id,
        album_type="album,single",
        limit=50
    )

    canciones = []
    for album in albums["items"]:
        album_tracks = sp.album_tracks(album["id"])
        for t in album_tracks["items"]:
            canciones.append(t["uri"])

    if canciones:
        tracks.extend(
            random.sample(canciones, min(canciones_por_artista, len(canciones)))
        )

random.shuffle(tracks)

# ========= BUSCAR PLAYLIST EXISTENTE =========
user_id = sp.current_user()["id"]
playlist_id = None

playlists = sp.current_user_playlists(limit=50)
for p in playlists["items"]:
    if p["name"] == PLAYLIST_NAME:
        playlist_id = p["id"]
        break

# ========= CREAR O LIMPIAR =========
if playlist_id:
    # vaciar playlist
    existing_tracks = sp.playlist_items(playlist_id, limit=100)
    uris = [item["track"]["uri"] for item in existing_tracks["items"] if item["track"]]
    if uris:
        sp.playlist_remove_all_occurrences_of_items(playlist_id, uris)
else:
    playlist = sp.user_playlist_create(
        user=user_id,
        name=PLAYLIST_NAME,
        public=False
    )
    playlist_id = playlist["id"]

# ========= AGREGAR =========
sp.playlist_add_items(playlist_id, tracks)

print(f"Playlist '{PLAYLIST_NAME}' actualizada con {len(tracks)} canciones")

¿Cuántos artistas random tomar? 3
¿Cuántas canciones random por artista? 2
Playlist 'Random desde TXT 🎲' actualizada con 6 canciones
